# Constellation v2 — Landmark training on Colab (Stage 1.5)

From-scratch 21-keypoint hand landmark regressor. Runs the same `aslv2.landmark.train`
code as local; only the device (CUDA) and data root differ.

## Before running — upload these **2 files** to one Drive folder (default `MyDrive/asl-landmark/`):
1. `colab_landmark_code.zip`  (from `model-v2/artifacts/`, run `python scripts/package_for_colab.py --landmark` to make it) — code + manifests
2. `landmark_small.zip`       (from `model-v2/artifacts/`, run `python scripts/shrink_landmark_for_colab.py` then zip the result) — shrunk FreiHAND images

**Runtime → Change runtime type → GPU (A100/L4/T4).**

See `model-v2/COLAB.md` → "Landmark (Stage-1.5)" for the full flow.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR  = '/content/drive/MyDrive/asl-landmark'   # <-- the folder you uploaded the 2 files to
DATA_ROOT  = '/content/data/landmark_small'
CODE_DIR   = '/content/model-v2'
assert os.path.isdir(DRIVE_DIR), f'Upload the 2 files to {DRIVE_DIR} first'
print('Drive folder contents:', os.listdir(DRIVE_DIR))

In [ ]:
# Copy zips from Drive to fast local Colab disk, then extract.
import shutil
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CODE_DIR,  exist_ok=True)

for z in ['landmark_small.zip', 'colab_landmark_code.zip']:
    src = f'{DRIVE_DIR}/{z}'
    print('copying', z, '...')
    shutil.copy(src, f'/content/{z}')
print('copied all zips to /content')

In [ ]:
# Extract: landmark_small.zip -> DATA_ROOT (freihand/training/... lands there)
!unzip -q -o /content/landmark_small.zip        -d /content
!unzip -q -o /content/colab_landmark_code.zip   -d $CODE_DIR

# Sanity: a manifest-relative path must resolve under DATA_ROOT
import json
m = json.load(open(f'{CODE_DIR}/artifacts/landmark/val_small.json'))
p = os.path.join(DATA_ROOT, m[0]['image'])
print('sample image resolves:', os.path.exists(p), '->', p)

In [ ]:
# Plan 6: COCO-WholeBody images — extract if uploaded alongside (optional)
import os, shutil
_cz = f'{DRIVE_DIR}/cocowb_landmark_small.zip'
if os.path.exists(_cz):
    shutil.copy(_cz, '/content/cocowb_landmark_small.zip')
    get_ipython().system('unzip -q -o /content/cocowb_landmark_small.zip -d /content')
    print("extracted COCO-WholeBody images")
else:
    print("no COCO-WholeBody zip in Drive folder — base-data-only run")


In [ ]:
!pip install -q -e $CODE_DIR
import torch
print('CUDA available:', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

In [ ]:
# Train. device() auto-selects CUDA on Colab.
# Checkpoints land in artifacts/checkpoints/landmark/.
!cd $CODE_DIR && python -m aslv2.landmark.train \
    --config configs/landmark_colab.yaml \
    --data-root $DATA_ROOT

In [ ]:
# Persist the trained landmark model back to Drive (so it survives the session).
ckpt = f'{CODE_DIR}/artifacts/checkpoints/landmark'
for f in ['best.pt', 'history.json']:
    src = f'{ckpt}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_DIR}/{f}')
        print('saved to Drive:', f)

import json
h = json.load(open(f'{ckpt}/history.json'))
print('best val PCK@0.2:', h.get('best_score'))
print('final epoch metrics:', h['history'][-1] if h.get('history') else None)
print('\nDownload best.pt from Drive into model-v2/artifacts/checkpoints/landmark/ on your machine.')